In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mariamgharibmenifii/hackathon-teamm/DeepX_train.xlsx
/kaggle/input/datasets/mariamgharibmenifii/hackathon-teamm/DeepX_validation.xlsx
/kaggle/input/datasets/mariamgharibmenifii/hackathon-teamm/DeepX_unlabeled.xlsx


In [26]:
df_train = pd.read_excel('/kaggle/input/datasets/mariamgharibmenifii/hackathon-teamm/DeepX_train.xlsx')
df_unlabeled = pd.read_excel('/kaggle/input/datasets/mariamgharibmenifii/hackathon-teamm/DeepX_unlabeled.xlsx')
df_validation = pd.read_excel('/kaggle/input/datasets/mariamgharibmenifii/hackathon-teamm/DeepX_validation.xlsx')


In [27]:
print("shape\n",df_train.shape)
print("describtion\n",df_train.describe())
print("nulls")
print(df_train.isnull().sum())

shape
 (1971, 9)
describtion
           review_id  star_rating
count   1971.000000  1971.000000
mean    5047.205479     3.341451
std     2800.399699     1.821920
min       16.000000     1.000000
25%     2646.000000     1.000000
50%     5122.000000     5.000000
75%     7377.500000     5.000000
max    10010.000000     5.000000
nulls
review_id            0
review_text          0
star_rating          0
date                 0
business_name        0
business_category    0
platform             0
aspects              0
aspect_sentiments    0
dtype: int64


In [29]:
df_train

,review_id,review_text,star_rating,date,business_name,business_category,platform,aspects,aspect_sentiments
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,2026-03-08 00:00:00,Noon,ecommerce,play_store,"[""app_experience"", ""delivery""]","{""app_experience"": ""negative"", ""delivery"": ""ne..."
1,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,قبل يومين (2),ممشي مصر Mawlana Cafe,كافيه,google_maps,"[""cleanliness"", ""ambiance"", ""service""]","{""cleanliness"": ""positive"", ""ambiance"": ""posit..."
2,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,1,قبل شهر,بيت لحم Beet Lahm,مطعم,google_maps,"[""service"", ""delivery"", ""food""]","{""service"": ""negative"", ""delivery"": ""negative""..."
3,3024,احلي مكان فزايد,5,قبل شهر,ذا بلكون كافيه الشيخ زايد,مطعم مأكولات ومشروبات,google_maps,"[""general""]","{""general"": ""positive""}"
4,5483,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,4,قبل سنة,The Best Restaurant,مطعم,google_maps,"[""food"", ""price""]","{""food"": ""positive"", ""price"": ""positive""}"
...,...,...,...,...,...,...,...,...,...
1966,6530,بصراحه كنت بحبو جدا جدا والناس كانت محترمه بس ...,1,2026-03-10 00:00:00,Careem,transport,play_store,"[""service"", ""delivery"", ""price"", ""app_experien...","{""service"": ""positive"", ""delivery"": ""negative""..."
1967,4612,المطعم قعدته حلوه وفى زحليقه ومرجيحه للاطفال\n...,3,قبل 3 أشهر,Samakmak Restaurant - 6th of October Branch,مطعم مأكولات بحرية,google_maps,"[""ambiance"", ""food"", ""price"", ""service""]","{""ambiance"": ""positive"", ""food"": ""neutral"", ""p..."
1968,4769,The worst experience at the HUB. Place is dirt...,1,قبل سنتين,ال ديفينو بيتزيريا,مطعم بيتزا,google_maps,"[""cleanliness"", ""food""]","{""cleanliness"": ""negative"", ""food"": ""negative""}"
1969,5366,الكوفي في فندق الهيلتون رمسيس...\n\nتم اضافه س...,1,قبل 4 سنوات,جاردن كورت كافيه,مقهى,google_maps,"[""service""]","{""service"": ""negative""}"


In [30]:
print(df_train.columns)

Index(['review_id', 'review_text', 'star_rating', 'date', 'business_name',
       'business_category', 'platform', 'aspects', 'aspect_sentiments'],
      dtype='object')


In [31]:
df_train['date']

0       2026-03-08 00:00:00
1             قبل يومين (2)
2                   قبل شهر
3                   قبل شهر
4                   قبل سنة
               ...         
1966    2026-03-10 00:00:00
1967             قبل 3 أشهر
1968              قبل سنتين
1969            قبل 4 سنوات
1970                قبل شهر
Name: date, Length: 1971, dtype: object

In [32]:
import pandas as pd
import re
from datetime import datetime

#refrence date
reference_date = pd.Timestamp.today()  # or pd.Timestamp.today()

#arabic normalize
def normalize_arabic(text):
    text = str(text)
    
    # normalize letters
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ى', 'ي', text)
    
    # remove brackets like (2)
    text = re.sub(r'\(.*?\)', '', text)
    
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

#tokenization
def tokenize(text):
    return text.split()

#semantic vocabulary
time_units = {
    # days
    "يوم": 1,
    "يومين": 2,
    "ايام": 1,
    
    # months
    "شهر": 30,
    "اشهر": 30,
    
    # years
    "سنه": 365,
    "سنين": 365,
    "سنتين": 2 * 365,
    "سنوات": 365
}

# extract number helper
def extract_number(tokens):
    for t in tokens:
        if t.isdigit():
            return int(t)
    return None

# NLP-based parsing
def parse_date_nlp(text):
    text = normalize_arabic(text)

    # absolute date ----
    try:
        dt = pd.to_datetime(text)
        return (reference_date - dt).days
    except:
        pass

    # relative Arabic ----
    tokens = tokenize(text)

    number = extract_number(tokens)
    unit_value = None

    for t in tokens:
        if t in time_units:
            unit_value = time_units[t]
            break

    if unit_value is not None:
        if number is None:
            number = 1
        return number * unit_value

    # fallback 
    return None

# apply on dataset
df_train['days_ago'] = df_train['date'].apply(parse_date_nlp)

# handle missing
df_train['days_ago'] = df_train['days_ago'].fillna(df_train['days_ago'].median())



In [36]:
import pandas as pd
import re
from datetime import timedelta

# reference date
reference_date = pd.Timestamp.today()  # or pd.Timestamp.today()

# normalize Arabic
def normalize_arabic(text):
    text = str(text)
    
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ى', 'ي', text)
    
    text = re.sub(r'\(.*?\)', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# tokenization
def tokenize(text):
    return text.split()

# time vocab (days)
time_units = {
    "يوم": 1,
    "يومين": 2,
    "ايام": 1,
    
    "شهر": 30,
    "اشهر": 30,
    
    "سنه": 365,
    "سنين": 365,
    "سنتين": 2 * 365,
    "سنوات": 365
}

# extract number
def extract_number(tokens):
    for t in tokens:
        if t.isdigit():
            return int(t)
    return None

# convert to DATETIME
def parse_to_datetime(text):
    text = normalize_arabic(text)

    # absolute date ----
    try:
        return pd.to_datetime(text)
    except:
        pass

    # relative Arabic ----
    tokens = tokenize(text)
    number = extract_number(tokens)
    unit_value = None

    for t in tokens:
        if t in time_units:
            unit_value = time_units[t]
            break

    if unit_value is not None:
        if number is None:
            number = 1
        
        days = number * unit_value
        return reference_date - timedelta(days=days)

    return pd.NaT

# apply
df_train['date_parsed'] = df_train['date'].apply(parse_to_datetime)

# extract features
df_train['year'] = df_train['date_parsed'].dt.year
df_train['month'] = df_train['date_parsed'].dt.month
df_train['day_of_week'] = df_train['date_parsed'].dt.dayofweek  # 0=Mon , 6=Sunday and so on

# handle missing
df_train['year'] = df_train['year'].fillna(df_train['year'].median())
df_train['month'] = df_train['month'].fillna(df_train['month'].median())
df_train['day_of_week'] = df_train['day_of_week'].fillna(df_train['day_of_week'].median())

In [45]:
df_train['date_parsed']

0      2026-03-08 00:00:00.000000
1      2026-04-22 08:29:19.564764
2      2026-03-25 08:29:19.564764
3      2026-03-25 08:29:19.564764
4      2025-04-24 08:29:19.564764
                  ...            
1966   2026-03-10 00:00:00.000000
1967   2026-01-24 08:29:19.564764
1968   2024-04-24 08:29:19.564764
1969   2022-04-25 08:29:19.564764
1970   2026-03-25 08:29:19.564764
Name: date_parsed, Length: 1971, dtype: datetime64[ns]

In [38]:
df_train['platform'].value_counts()

platform
google_maps    1210
play_store      761
Name: count, dtype: int64

In [39]:
df_train = pd.get_dummies(df_train, columns=['platform'])

In [43]:
print(type(df_train))

<class 'pandas.core.frame.DataFrame'>


In [46]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1971 entries, 0 to 1970
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   review_id             1971 non-null   int64         
 1   review_text           1971 non-null   object        
 2   star_rating           1971 non-null   int64         
 3   date                  1971 non-null   object        
 4   business_name         1971 non-null   object        
 5   business_category     1971 non-null   object        
 6   aspects               1971 non-null   object        
 7   aspect_sentiments     1971 non-null   object        
 8   days_ago              1971 non-null   float64       
 9   date_parsed           1606 non-null   datetime64[ns]
 10  year                  1971 non-null   float64       
 11  month                 1971 non-null   float64       
 12  day_of_week           1971 non-null   float64       
 13  platform_google_ma

In [47]:
df_train['platform_google_maps']

0       False
1        True
2        True
3        True
4        True
        ...  
1966    False
1967     True
1968     True
1969     True
1970     True
Name: platform_google_maps, Length: 1971, dtype: bool

In [48]:
df_train['platform_play_store']

0        True
1       False
2       False
3       False
4       False
        ...  
1966     True
1967    False
1968    False
1969    False
1970    False
Name: platform_play_store, Length: 1971, dtype: bool

In [49]:
def identify_outliers(column_values):
  plt.figure(figsize=(8, 4))
  sns.boxplot(x=column_values, color='lightgreen')
  plt.title('Box Plot to Identify Outliers and Symmetry')
  plt.show()

In [53]:
#boolean ti int
df_train['platform_google_maps'] = df_train['platform_google_maps'].astype(int)
df_train['platform_play_store'] = df_train['platform_play_store'].astype(int)

print(df_train['platform_google_maps'].isnull().sum())
print(df_train['platform_play_store'].isnull().sum())

print(df_train['platform_google_maps'].value_counts())
print(df_train['platform_play_store'].value_counts())

0
0
platform_google_maps
1    1210
0     761
Name: count, dtype: int64
platform_play_store
0    1210
1     761
Name: count, dtype: int64
